# scibert

In [ ]:
# =====پیش‌نیازها =====
!pip install TorchCRF pytorch-crf
!pip install --quiet transformers datasets seqeval torch evaluate torchcrf

In [ ]:
# ===== کتابخانه‌ها =====
import os, json
import torch
import numpy as np
import pandas as pd
from collections import Counter
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModel, TrainingArguments, Trainer
from torchcrf import CRF
import torch.nn as nn
import torch.nn.functional as F
from torch.nn import TransformerEncoder, TransformerEncoderLayer
from torch.utils.data import DataLoader
from dataclasses import dataclass
from typing import List, Dict, Any
from seqeval.metrics import classification_report, precision_score, recall_score, f1_score

# ===== نام مدل =====
MODEL_NAME = "allenai/scibert_scivocab_cased"

if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.cuda.empty_cache()
    
def read_iob_file(file_path):
    sentences, labels = [], []
    tokens, tags = [], []
    with open(file_path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                if tokens:
                    sentences.append(tokens)
                    labels.append(tags)
                    tokens, tags = [], []
            else:
                parts = line.split()
                tok, tag = parts[0], parts[-1]
                tokens.append(tok)
                tags.append(tag)
        if tokens:
            sentences.append(tokens)
            labels.append(tags)
    return sentences, labels

# مسیر فایل‌ها
train_file = "/content/drive/MyDrive/poroje/data/train_NER_Tech.iob"
test_file  = "/content/drive/MyDrive/poroje/data/test_NER_Tech.iob"

train_sents, train_tags = read_iob_file(train_file)
test_sents, test_tags = read_iob_file(test_file)
unique_labels = sorted({t for seq in train_tags + test_tags for t in seq})
combined_labels = ["O"] + [l for l in unique_labels if l != "O"]
label2id = {l: i for i, l in enumerate(combined_labels)}
id2label = {i: l for l, i in label2id.items()}

flat_tags = [t for seq in train_tags for t in seq]
counts = Counter(flat_tags)
total = sum(counts.values())
num_lbls = len(combined_labels)
class_weights = {
    lbl: total / (num_lbls * counts.get(lbl, 1))
    for lbl in combined_labels
}
weights_tensor = torch.tensor(
    [class_weights[lbl] for lbl in combined_labels],
    dtype=torch.float
)
for cls in ["I-TASK", "I-DATASET", "B-TECHNOLOGY", "I-TECHNOLOGY"]:
    if cls in label2id:
        weights_tensor[label2id[cls]] *= 2.0

train_examples = [
    {"tokens": s, "ner_tags": [label2id[t] for t in ts]}
    for s, ts in zip(train_sents, train_tags)
]
eval_examples = [
    {"tokens": s, "ner_tags": [label2id[t] for t in ts]}
    for s, ts in zip(test_sents, test_tags)
]
datasets = DatasetDict({
    "train": Dataset.from_list(train_examples),
    "eval": Dataset.from_list(eval_examples)
})
print(datasets)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
MAX_LEN = 256

def tokenize_and_align_iob(ex):
    enc = tokenizer(
        ex["tokens"],
        is_split_into_words=True,
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN,
        add_special_tokens=False,
        return_offsets_mapping=True
    )
    word_ids = enc.word_ids()
    enc.pop("offset_mapping")
    labels_aln, token_mask = [], []
    prev = None
    for wi in word_ids:
        if wi is None:
            labels_aln.append(label2id["O"])
            token_mask.append(0)
        elif wi != prev:
            labels_aln.append(ex["ner_tags"][wi])
            token_mask.append(1)
        else:
            tag = combined_labels[ex["ner_tags"][wi]]
            if tag.startswith("B-"):
                tag = tag.replace("B-", "I-", 1)
            labels_aln.append(label2id[tag])
            token_mask.append(1)
        prev = wi
    enc["labels"] = labels_aln
    enc["token_mask"] = token_mask
    return enc

tokenized = datasets.map(
    tokenize_and_align_iob,
    batched=False,
    remove_columns=["tokens", "ner_tags"]
)
print(tokenized)

@dataclass
class MyTokenCollator:
    tokenizer: Any
    label_pad_token_id: int

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        labels = [f.pop("labels") for f in features]
        masks = [f.pop("token_mask") for f in features]
        batch = self.tokenizer.pad(features, return_tensors="pt")
        max_len = batch["input_ids"].size(1)

        def pad_to(x, pad_val, L):
            return x + [pad_val] * (L - len(x))

        batch["labels"] = torch.tensor(
            [pad_to(l, self.label_pad_token_id, max_len) for l in labels],
            dtype=torch.long
        )
        batch["token_mask"] = torch.tensor(
            [pad_to(m, 0, max_len) for m in masks],
            dtype=torch.bool
        )
        return batch

data_collator = MyTokenCollator(
    tokenizer,
    label_pad_token_id=label2id["O"]
)

class BertCRFForTokenClassification(nn.Module):
    def __init__(
        self,
        model_name: str,
        num_labels: int,
        labels_vocab,
        feature_extraction: bool = False,
        class_weights: torch.Tensor = None,
        focal_alpha: float = 2.0,
        focal_gamma: float = 3.0
    ):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        if feature_extraction:
            for p in self.bert.parameters():
                p.requires_grad = False

        self.dropout1 = nn.Dropout(0.3)
        self.dropout2 = nn.Dropout(0.3)
        self.word_transformer_layer = TransformerEncoderLayer(
            d_model=self.bert.config.hidden_size,
            nhead=8,
            dim_feedforward=self.bert.config.hidden_size * 4,
            batch_first=True
        )
        self.word_transformer = TransformerEncoder(
            self.word_transformer_layer,
            num_layers=1
        )
        self.classifier = nn.Linear(self.bert.config.hidden_size, num_labels)
        self.crf = CRF(num_labels, batch_first=True)
        self.class_weights = class_weights
        self.focal_alpha = focal_alpha
        self.focal_gamma = focal_gamma
        self.labels_vocab = labels_vocab
        self._apply_bio_constraints()

    def _apply_bio_constraints(self):
        L = self.labels_vocab

        def typ(lbl):
            return lbl.split("-", 1)[1] if lbl != "O" else ""

        with torch.no_grad():
            for i, li in enumerate(L):
                if li.startswith("I-"):
                    self.crf.start_transitions[i] = -1e4
                for j, lj in enumerate(L):
                    if lj.startswith("I-"):
                        ok_prev = (
                            li == f"B-{typ(lj)}"
                            or li == f"I-{typ(lj)}"
                        )
                        if not ok_prev:
                            self.crf.transitions[i, j] = -1e4

    def forward(
        self,
        input_ids,
        attention_mask=None,
        token_type_ids=None,
        labels=None,
        token_mask=None
    ):
        x = self.bert(
            input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids
        ).last_hidden_state
        x = self.dropout1(x)
        valid_mask = token_mask.bool() if token_mask is not None else attention_mask.bool()
        x = self.word_transformer(x, src_key_padding_mask=~valid_mask)
        x = self.dropout2(x)
        emissions = self.classifier(x)

        if labels is not None:
            b, s, n = emissions.size()
            em = emissions.reshape(-1, n)[valid_mask.reshape(-1)]
            lb = labels.reshape(-1)[valid_mask.reshape(-1)]
            ce = F.cross_entropy(
                em,
                lb,
                weight=self.class_weights.to(em.device) if self.class_weights is not None else None,
                reduction="none"
            )
            pt = torch.exp(-ce)
            focal = (self.focal_alpha * (1 - pt) ** self.focal_gamma * ce).mean()
            crf_loss = -self.crf(
                emissions,
                labels,
                mask=valid_mask,
                reduction="mean"
            )
            return {"loss": crf_loss + focal, "logits": emissions}

        best_paths = self.crf.decode(emissions, mask=valid_mask)
        max_len = emissions.size(1)
        padded = [seq + [0] * (max_len - len(seq)) for seq in best_paths]
        return torch.tensor(padded, device=emissions.device)

# ===== آموزش =====
EPOCHS = 30
CHECKPOINT_DIR = "./temp"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

model = BertCRFForTokenClassification(
    MODEL_NAME,
    len(combined_labels),
    labels_vocab=combined_labels,
    feature_extraction=False,
    class_weights=weights_tensor,
    focal_alpha=2.0,
    focal_gamma=3.0
)

training_args = TrainingArguments(
    output_dir=CHECKPOINT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    learning_rate=1e-5,
    weight_decay=0.1,
    warmup_ratio=0.3,
    logging_dir="./logs",
    logging_steps=100,
    save_steps=200,
    save_total_limit=2,
    do_train=True,
    do_eval=True,
    report_to=[]
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["eval"],
    data_collator=data_collator,
    tokenizer=tokenizer
)

trainer.train()
eval_results = trainer.evaluate()
print("\n=== Eval (HF; loss-based) ===")
print(eval_results)
eval_loader = DataLoader(
    tokenized["eval"],
    batch_size=32,
    collate_fn=data_collator,
    pin_memory=True,
    num_workers=2
)

model.eval()
device = next(model.parameters()).device
all_true, all_pred = [], []

with torch.no_grad():
    for batch in eval_loader:
        labels = batch.pop("labels")
        tmask = batch["token_mask"]
        batch = {k: v.to(device) for k, v in batch.items()}
        decoded_ids = model(**batch).cpu()

        for true_ids, pred_ids, m in zip(labels, decoded_ids, tmask):
            seq_true, seq_pred = [], []
            for t_id, p_id, mk in zip(true_ids, pred_ids, m):
                if mk:
                    seq_true.append(id2label[int(t_id)])
                    seq_pred.append(id2label[int(p_id)])
            all_true.append(seq_true)
            all_pred.append(seq_pred)

print("\n--- Classification Report ---")
print(classification_report(all_true, all_pred, digits=4))
print(f"\nOverall Precision: {precision_score(all_true, all_pred):.4f}")
print(f"Overall Recall:    {recall_score(all_true, all_pred):.4f}")
print(f"Overall F1:        {f1_score(all_true, all_pred):.4f}")

DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 55401
    })
    eval: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 13973
    })
})


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/55401 [00:00<?, ? examples/s]

...

/tmp/ipython-input-1382710504.py:328: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
You're using a BertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Step,Training Loss
63600,0.103700
63700,0.114800
63800,0.132900
63900,0.103900
64000,0.066300
64100,0.081200
64200,0.110500
64300,0.145300
64400,0.089100
64500,0.104700


Step,Training Loss
63600,0.103700
63700,0.114800
63800,0.132900
63900,0.103900
64000,0.066300
64100,0.081200
64200,0.110500
64300,0.145300
64400,0.089100
64500,0.104700



=== Eval (loss-based) ===


{'eval_loss': 7.961144924163818, 'eval_runtime': 101.1761, 'eval_samples_per_second': 138.106, 'eval_steps_per_second': 4.319, 'epoch': 40.0}

--- Classification Report (CRF decode + Standard Smart Switch v2) ---
              precision    recall  f1-score   support

     DATASET     0.4256    0.7608    0.5459       485
      METHOD     0.4719    0.4884    0.4800       344
      METRIC     0.9576    0.3152    0.4743      1361
       MODEL     0.8948    0.8586    0.8763      4716
        TASK     0.6459    0.7469    0.6927      1995
  TECHNOLOGY     0.7466    0.8851    0.8100      5151
       TOOLS     0.6719    0.4216    0.5181      3031

   micro avg     0.7475    0.7225    0.7348     17083
   macro avg     0.6878    0.6395    0.6282     17083
weighted avg     0.7647    0.7225    0.7219     17083


Overall Precision: 0.7475
Overall Recall:    0.7225
Overall F1:        0.7348

--- Confusion Matrix ---
                   O  B-DATASET  B-METHOD  B-METRIC  B-MODEL  B-TASK  \
O            

# پاک سازی

In [ ]:
import os, sys, glob, pathlib

print("CWD:", os.getcwd())
print("sys.path (top 3):", sys.path[:3])
print("Suspicious entries in CWD:", [p for p in os.listdir() if p.lower().startswith("torch")])

for pat in ["./torch.py", "./torch", "./**/torch.py", "./**/torch"]:
    print(pat, "->", glob.glob(pat, recursive=True)[:10])


CWD: /content
sys.path (top 3): ['/content', '/env/python', '/usr/lib/python312.zip']
Suspicious entries in CWD: []
./torch.py -> []
./torch -> []
./**/torch.py -> []
./**/torch -> []


In [ ]:
import shutil, os

for fp in ["torch.py", "torch.pyc", "__pycache__/torch.cpython-*.pyc"]:
    if os.path.exists(fp):
        os.remove(fp)

if os.path.isdir("torch"):
    shutil.rmtree("torch", ignore_errors=True)

# اگر داخل پروژه‌ات پوشه‌ای به اسم torch هست، مثلاً:
bad_dir = "/content/drive/MyDrive/poroje/torch"
if os.path.isdir(bad_dir):
    shutil.rmtree(bad_dir, ignore_errors=True)


In [ ]:
import shutil, os, glob
for d in glob.glob("./**/__pycache__", recursive=True):
    shutil.rmtree(d, ignore_errors=True)


In [ ]:
import IPython
IPython.Application.instance().kernel.do_shutdown(restart=True)


{'status': 'ok', 'restart': True}

In [ ]:
import IPython
IPython.Application.instance().kernel.do_shutdown(restart=True)


{'status': 'ok', 'restart': True}